<h1 style="text-align: center; font-size: 50px;"> 🌍 Word Embeddings Generation</h1>

This Jupyter notebook demonstrates how to generate word embeddings from a given corpus using a pre-trained BERT model. These embeddings will be used to find semantically similar matches for a user query.

# Notebook Overview
- Install and Import Libraries
- Start Execution
- Configure Settings
- Verify Assets
- Load and Preprocess Data
- Initialize BERT Tokenizer and Model
- Generate Embeddings in Batches
- Save Embeddings to File
- Downloading the Bert Large Uncased Model

In [1]:
%%time

%pip install -r ../requirements.txt --quiet

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_utilities-0.14.0-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/deep_ep-1.0.0+a84a248-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/looseversion-1.3.0-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is depreca

In [2]:
MIN_TOTAL_RAM_GB = 16
MIN_TOTAL_VRAM_GB = 4


from ai_studio_blueprint_kit.memory_guard import run_memory_check_notebook


run_memory_check_notebook(
    min_total_ram_gb=MIN_TOTAL_RAM_GB,
    min_total_vram_gb=MIN_TOTAL_VRAM_GB,
)

# Start Execution

In [3]:
import logging  # For application-level logging
import time     # For runtime measurement (wall clock)

# Configure logger
logger: logging.Logger = logging.getLogger("run_workflow_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [4]:
start_time = time.time()  

logger.info("Notebook execution started.")

2026-04-20 15:18:46 - INFO - Notebook execution started.


# Install and Import Libraries

In [5]:
# -----------------------------
# Standard library imports
# -----------------------------
import os                   # Operating system utilities (paths, env vars, etc.)
import sys                  # Python runtime environment manipulation
import warnings             # Warning control and message handling
from datetime import datetime  # Date and time handling
from pathlib import Path     # Object-oriented filesystem paths
import time

# -----------------------------
# Data manipulation libraries
# -----------------------------
import numpy as np           # Numerical computations and arrays
import pandas as pd          # Data manipulation and analysis
from sklearn.metrics.pairwise import cosine_similarity  # Pairwise similarity metrics
from tabulate import tabulate  # Pretty-print tabular data

# -----------------------------
# Deep learning frameworks
# -----------------------------
import torch                 # PyTorch deep learning framework

# -----------------------------
# NLP libraries
# -----------------------------
import nltk                  # Natural Language Toolkit (tokenization, corpora, etc.)
from nemo.collections.nlp.models import BERTLMModel  # NVIDIA NeMo BERT language model
from transformers import AutoTokenizer  # Tokenizer for transformer-based models
from transformers import logging as hf_logging
import mlflow
from mlflow import MlflowClient
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec, TensorSpec, ParamSchema, ParamSpec
from mlflow.tracking import MlflowClient

# Define the relative path to the 'src' directory
src_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if src_path not in sys.path:
    sys.path.append(src_path)

from src.utils import load_config, download_from_s3_uri

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[NeMo W 2026-04-20 15:19:08 nemo_logging:405] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
      warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
    


# Configure Settings

In [6]:
# ------------------------ Suppress Verbose Logs ------------------------
warnings.filterwarnings("ignore")

In [7]:
# Load configuration
config = load_config("../configs/config.yaml")

DATA_NAME = "corpus.csv"
DATA_URI = f"s3://149536453923-hpaistudio-public-assets/AI-Blueprints/ngc-integration/vacation-recommendation-with-bert/{DATA_NAME}"
CORPUS_PATH = "../data/raw/corpus.csv"
TOKENIZER_DIR = "../artifacts/tokenizer"
BERT_MODEL_NAME = "bert-large-uncased"
BERT_MODEL_DATAFABRIC_PATH = config.get("model_path", "/home/jovyan/datafabric/Bertlargeuncased/bertlargeuncased.nemo")
EMBEDDINGS_OUTPUT_PATH = "../data/processed/"
BERT_MODEL_ONLINE_PATH = "/root/.cache/torch/NeMo/NeMo_1.22.0/bertlargeuncased/ca4ebba9f05a8ffb79845249ca046983/bertlargeuncased.nemo"
DEMO_PATH = "../demo"
EMBEDDINGS_PATH = "../data/processed/embeddings.csv"
MODEL_NAME = "BERT_Tourism_Model"

In [8]:
CORPUS_PATH = "../data/raw/corpus.csv"
TOKENIZER_DIR = "../artifacts/tokenizer"
BERT_MODEL_NAME = "bert-large-uncased"
BERT_MODEL_DATAFABRIC_PATH = "/home/jovyan/datafabric/Bertlargeuncased/bertlargeuncased.nemo"
EMBEDDINGS_OUTPUT_PATH = "../data/processed/"
BERT_MODEL_ONLINE_PATH = "/root/.cache/torch/NeMo/NeMo_1.22.0/bertlargeuncased/ca4ebba9f05a8ffb79845249ca046983/bertlargeuncased.nemo"
DEMO_PATH = "../demo"
EMBEDDINGS_PATH = "../data/processed/embeddings.csv"
MODEL_NAME = "BERT_Tourism_Model"

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [10]:
%%time

saved_data = download_from_s3_uri(s3_uri=DATA_URI, local_path="../data/raw")
logger.info(f"Saved to: {saved_data}")

2026-04-20 15:19:13 - INFO - Saved to: /home/jovyan/AI-Blueprints/ngc-integration/vacation-recommendation-with-bert/data/raw/corpus.csv


CPU times: user 78.7 ms, sys: 65 ms, total: 144 ms
Wall time: 1.3 s


# Verify Assets

In [11]:
def log_asset_status(asset_path: str, asset_name: str, success_message: str, failure_message: str) -> None:
    """
    Logs the status of a given asset based on its existence.

    Parameters:
        asset_path (str): File or directory path to check.
        asset_name (str): Name of the asset for logging context.
        success_message (str): Message to log if asset exists.
        failure_message (str): Message to log if asset does not exist.
    """
    if Path(asset_path).exists():
        logger.info(f"{asset_name} is properly configured. {success_message}")
    else:
        logger.error(f"{asset_name} is not properly configured. {failure_message}")

log_asset_status(
    asset_path=BERT_MODEL_DATAFABRIC_PATH ,
    asset_name="BERT model",
    success_message="",
    failure_message="Please download the required assets in your project on AI Studio."
)

log_asset_status(
    asset_path=CORPUS_PATH,
    asset_name="Corpus data",
    success_message="",
    failure_message="Please check if Corpus was properly downloaded in your project on AI Studio."
)

2026-04-20 15:19:13 - INFO - BERT model is properly configured. 
2026-04-20 15:19:13 - INFO - Corpus data is properly configured. 


# Load and Preprocess Data

In [12]:
%%time

# Download the Punkt tokenizer data for sentence tokenization
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...


CPU times: user 178 ms, sys: 36.9 ms, total: 215 ms
Wall time: 801 ms


[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [13]:
# Load the dataset into a Pandas DataFrame
corpus_df = pd.read_csv(CORPUS_PATH)

# Display the first few rows of the DataFrame
logger.info("First few entries of the DataFrame:")
print(corpus_df.head())

2026-04-20 15:19:14 - INFO - First few entries of the DataFrame:


   Unnamed: 0  Topic                                             Pledge
0           0      1  Actually we as an association are still pretty...
1           1      1  EFFAT welcomes the Commission Proposal for a R...
2           2      1  HOTREC calls for a level playing field and fai...
3           3      1  Estonia sees the need to synchronize and harmo...
4           4      1  Sphere Travel Club contributes to a flourishin...


In [14]:
documents = corpus_df["Pledge"].astype(str).tolist()  # Convert the column to a list

# Initialize BERT Tokenizer and Model

In [15]:
%%time

# Initialize the tokenizer with a pre-trained BERT model
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
tokenizer.save_pretrained(TOKENIZER_DIR)

# Set device to GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

logger.info("Loading BERT model...")

# Ensure you have added the 'bertlargeuncased' model from the NVIDIA NGC model catalog.
# If unavailable, use the alternative method below to download the model online.

# Uncomment the following line to download the BERT model online:
# bert_model = BERTLMModel.from_pretrained(model_name="bertlargeuncased", strict=False).to(device)

# Load the BERT model from a local .nemo file inside datafabric folder
bert_model = BERTLMModel.restore_from(BERT_MODEL_DATAFABRIC_PATH, strict=False).to(device)

logger.info("BERT model loaded successfully.")

2026-04-20 15:19:14 - INFO - Loading BERT model...
[NeMo W 2026-04-20 15:20:06 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    data_file: /home/yzhang/data/nlp/bert/47316/hdf5/lower_case_1_seq_len_512_max_pred_80_masked_lm_prob_0.15_random_seed_12345_dupe_factor_5_shard_1472_test_split_10/books_wiki_en_corpus/training/
    max_predictions_per_seq: 80
    batch_size: 16
    shuffle: true
    num_samples: -1
    num_workers: 2
    drop_last: false
    pin_memory: false
    
[NeMo W 2026-04-20 15:20:06 nemo_logging:405] bert-large-uncased is not in get_pretrained_lm_models_list(include_external=False), will be using AutoModel from HuggingFace.
[NeMo W 2026-04-20 15:20:32 nemo_logging:405] Trainer wasn't specified in model constructor. Make sure that you really wanted it.


[NeMo I 2026-04-20 15:20:32 nemo_logging:393] Optimizer config = AdamW (
    Parameter Group 0
        amsgrad: False
        betas: (0.9, 0.999)
        capturable: False
        decoupled_weight_decay: True
        differentiable: False
        eps: 1e-08
        foreach: None
        fused: None
        lr: 4.375e-05
        maximize: False
        weight_decay: 0.01
    )


[NeMo W 2026-04-20 15:20:32 nemo_logging:405] Neither `max_steps` nor `iters_per_batch` were provided to `optim.sched`, cannot compute effective `max_steps` !
    Scheduler will not be instantiated !


[NeMo I 2026-04-20 15:20:34 nemo_logging:393] Model BERTLMModel was successfully restored from /home/jovyan/datafabric/Bertlargeuncased/bertlargeuncased.nemo.


2026-04-20 15:20:34 - INFO - BERT model loaded successfully.


CPU times: user 19.6 s, sys: 9.87 s, total: 29.5 s
Wall time: 1min 19s


# Generate Embeddings in Batches

In [16]:
def generate_embeddings_in_batches(texts, tokenizer, model, batch_size=32):
    """
    Generates text embeddings using the NeMo BERT model in batches.
    
    Args:
        texts (list of str): List of input texts.
        tokenizer: Pretrained tokenizer.
        model: Pretrained NeMo BERT model.
        batch_size (int, optional): Batch size for processing. Default is 32.
    
    Returns:
        np.ndarray: Generated embeddings.
    """
    model.eval()  # Set model to evaluation mode
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        
        # Tokenize batch with padding and truncation
        encoded_input = tokenizer(
            batch_texts, padding=True, truncation=True, return_tensors="pt", max_length=128
        )
        encoded_input = {key: val.to(device) for key, val in encoded_input.items()}

        with torch.no_grad():  # Disable gradient computation for inference
            output = model.bert_model(**encoded_input)
        
        # Extract the CLS token representation for embeddings
        embeddings = output[:, 0, :].cpu().numpy()  # CLS token representation
        all_embeddings.append(embeddings)

    return np.vstack(all_embeddings)

# Save Embeddings to File

In [17]:
%%time

# Generate embeddings using the pre-trained model
embeddings = generate_embeddings_in_batches(documents, tokenizer, bert_model)

# Convert embeddings into a DataFrame
df_embeddings = pd.DataFrame(embeddings)

# Ensure the output directory exists
os.makedirs(EMBEDDINGS_OUTPUT_PATH, exist_ok=True)
    
# Define output file path
output_file = os.path.join(EMBEDDINGS_OUTPUT_PATH, "embeddings.csv")

# Save embeddings
df_embeddings.to_csv(output_file , index=False)

logger.info(f"✅ Embedding completed and saved to: {output_file}")

2026-04-20 15:20:53 - INFO - ✅ Embedding completed and saved to: ../data/processed/embeddings.csv


CPU times: user 18 s, sys: 5.61 s, total: 23.6 s
Wall time: 19.5 s


In [18]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")

2026-04-20 15:20:53 - INFO - ⏱️ Total execution time: 2m 6.68s


In [19]:
print("Notebook execution completed successfully.")

Notebook execution completed successfully.


Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).